In [4]:
import os
import pickle
from pathlib import Path
import numpy as np

# Initialize lists for training and test data
train_data = []
train_labels = []
test_data = []
test_labels = []

# Define the base path
base_path = Path('/lambda/nfs/nba-rapture/nba_data_v2/Regular season/box')

# Loop through each player folder in box
for player_folder in base_path.iterdir():
    if player_folder.is_dir():
        tensor_file = player_folder / 'tensor.pkl'
        
        # Check if tensor.pkl exists
        if tensor_file.exists():
            print(f"Processing: {player_folder.name}")
            
            # Load the pickle file
            with open(tensor_file, 'rb') as f:
                tensor_dict = pickle.load(f)
            
            # Loop through each timestamp in the dictionary
            for timestamp, data_dict in tensor_dict.items():
                # Check if timestamp is historical (< '20201101000000')
                if timestamp <= '20201101000000':
                    # Historical data goes to test sets
                    test_data.append(data_dict['data'])
                    test_labels.append(float(data_dict['label']['rap_box']))
                else:
                    # Non-historical data goes to training sets
                    train_data.append(data_dict['data'])
                    train_labels.append(float(data_dict['label']['rap_box']))

# Print summary statistics
print(f"\nProcessing complete!")
print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Total samples: {len(train_data) + len(test_data)}")

# Check for inconsistent lengths
train_lengths = [len(x) for x in train_data]
test_lengths = [len(x) for x in test_data]
print(f"\nBefore truncation:")
print(f"Train data lengths: min={min(train_lengths)}, max={max(train_lengths)}")
print(f"Test data lengths: min={min(test_lengths)}, max={max(test_lengths)}")

# Find the minimum length across both train and test
min_length = min(min(train_lengths), min(test_lengths))
print(f"\nMinimum length across all data: {min_length}")

# Truncate all data to the minimum length
train_data_truncated = [x[:min_length] for x in train_data]
test_data_truncated = [x[:min_length] for x in test_data]

# Convert to numpy arrays
train_data_array = np.array(train_data_truncated)
train_labels_array = np.array(train_labels)
test_data_array = np.array(test_data_truncated)
test_labels_array = np.array(test_labels)

# Print final statistics
print(f"\nAfter truncation:")
print(f"Train data shape: {train_data_array.shape}")
print(f"Train labels shape: {train_labels_array.shape}")
print(f"Test data shape: {test_data_array.shape}")
print(f"Test labels shape: {test_labels_array.shape}")


Processing: Carmelo Anthony
Processing: Reggie Perry
Processing: Thon Maker
Processing: Quinn Cook
Processing: Danilo Gallinari
Processing: Dante Exum
Processing: Dewayne Dedmon
Processing: Jaylen Nowell
Processing: Duncan Robinson
Processing: Georges Niang
Processing: Robin Lopez
Processing: David Lee
Processing: Kevin Martin
Processing: John Collins
Processing: Moses Brown
Processing: Tarik Black
Processing: Josh McRoberts
Processing: Richaun Holmes
Processing: Isaiah Taylor
Processing: Maxi Kleber
Processing: Robert Williams III
Processing: Terry Rozier
Processing: Derrick White
Processing: Isaac Okoro
Processing: Rui Hachimura
Processing: Moritz Wagner
Processing: Stephen Curry
Processing: Jamal Murray
Processing: Theo Maledon
Processing: Gordon Hayward
Processing: KZ Okpala
Processing: Vernon Carey Jr.
Processing: Abdel Nader
Processing: Mike Dunleavy
Processing: Royce O'Neale
Processing: Marvin Bagley III
Processing: Kent Bazemore
Processing: Brandon Bass
Processing: Richard Jeff

In [2]:
# !pip install pymongo

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 KB 69.6 MB/s eta 0:00:00


In [2]:
# # manual filtering for rap_box_o only
# good_length = 405
# new_train_data = [x for x in train_data if len(x)==good_length]
# new_train_labels = [train_labels[j] for j in range(len(train_data)) if len(train_data[j])==good_length]
# new_test_data = [x for x in test_data if len(x)==good_length]
# new_test_labels = [test_labels[j] for j in range(len(test_data)) if len(test_data[j])==good_length]

In [5]:
# print(new_train_data)
# print(new_train_labels)
# print(new_test_data)
# print(new_test_labels)
train_data = train_data_array
train_labels = train_labels_array
test_data = test_data_array
test_labels = test_labels_array

In [6]:
import numpy as np

def deduplicate_data(data_list, labels_list):
    """
    Remove duplicates from data and corresponding labels.
    
    Args:
        data_list: List of data vectors
        labels_list: List of corresponding labels
    
    Returns:
        Tuple of (deduplicated_data, deduplicated_labels)
    """
    # Convert each data item to tuple for hashing
    seen = {}
    unique_data = []
    unique_labels = []
    
    for i, (data, label) in enumerate(zip(data_list, labels_list)):
        # Convert data to tuple (hashable) for comparison
        # If data is already a numpy array, convert to tuple
        if isinstance(data, np.ndarray):
            data_key = tuple(data.flatten())
        else:
            data_key = tuple(data) if not isinstance(data, tuple) else data
        
        # Only add if we haven't seen this data before
        if data_key not in seen:
            seen[data_key] = True
            unique_data.append(data)
            unique_labels.append(label)
    
    return unique_data, unique_labels

# Get original counts
print(f"Original training samples: {len(train_data)}")
print(f"Original test samples: {len(test_data)}")

# De-duplicate training data
train_data_dedup, train_labels_dedup = deduplicate_data(train_data, train_labels)

# De-duplicate test data
test_data_dedup, test_labels_dedup = deduplicate_data(test_data, test_labels)

# Print results
print(f"\nAfter de-duplication:")
print(f"Training samples: {len(train_data_dedup)} (removed {len(train_data) - len(train_data_dedup)} duplicates)")
print(f"Test samples: {len(test_data_dedup)} (removed {len(test_data) - len(test_data_dedup)} duplicates)")

# Replace original lists with deduplicated versions
train_data = train_data_dedup
train_labels = train_labels_dedup
test_data = test_data_dedup
test_labels = test_labels_dedup

Original training samples: 72154
Original test samples: 1349

After de-duplication:
Training samples: 31296 (removed 40858 duplicates)
Test samples: 1349 (removed 0 duplicates)


In [17]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import os
from datetime import datetime
import csv

# Set random seed for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# ============================================================================
# DATA LOADING - MODIFIED FOR 403 FEATURES
# ============================================================================
print("Loading data...")
print("Expected features: 403 (box scores)")

# Convert lists to numpy arrays
train_data_array = np.vstack(train_data)
test_data_array = np.vstack(test_data)

train_labels_array = np.array(train_labels, dtype=np.float32).reshape(-1, 1)
test_labels_array = np.array(test_labels, dtype=np.float32).reshape(-1, 1)

print(f"Original data shape: {train_data_array.shape}")
print(f"Original labels shape: {train_labels_array.shape}")

# Verify feature count
expected_features = 403
actual_features = train_data_array.shape[1]
if actual_features != expected_features:
    print(f"\n⚠️  WARNING: Expected {expected_features} features but got {actual_features}")
    print(f"    Proceeding with {actual_features} features...")
else:
    print(f"✓ Feature count confirmed: {actual_features}")

# ============================================================================
# LABEL ANALYSIS AND TRANSFORMATION
# ============================================================================
print("\n" + "="*70)
print("LABEL ANALYSIS")
print("="*70)

all_labels = np.concatenate([train_labels_array, test_labels_array])
min_label = np.min(all_labels)
max_label = np.max(all_labels)

print(f"Original label distribution:")
print(f"  Min: {min_label:.6f}")
print(f"  Max: {max_label:.6f}")
print(f"  Mean: {np.mean(all_labels):.6f}")
print(f"  Median: {np.median(all_labels):.6f}")
print(f"  Std: {np.std(all_labels):.6f}")

# Apply shift
if min_label < 0:
    label_shift = abs(min_label) + 0.001
else:
    label_shift = 0.001 - min_label

train_labels_array = train_labels_array + label_shift
test_labels_array = test_labels_array + label_shift

all_labels_shifted = np.concatenate([train_labels_array, test_labels_array])
new_min = np.min(all_labels_shifted)
new_max = np.max(all_labels_shifted)

print(f"\nShift applied: +{label_shift:.6f}")
print(f"Transformed range: [{new_min:.6f}, {new_max:.6f}]")
print("="*70 + "\n")

# ============================================================================
# FEATURE PREPROCESSING
# ============================================================================
print("Feature preprocessing...")

input_size = train_data_array.shape[1]
print(f"Original input features: {input_size}")

# Handle missing values
train_data_array = np.nan_to_num(train_data_array, nan=0.0, posinf=0.0, neginf=0.0)
test_data_array = np.nan_to_num(test_data_array, nan=0.0, posinf=0.0, neginf=0.0)

# Remove constant features
feature_std = np.std(train_data_array, axis=0)
non_constant_features = feature_std > 1e-6
train_data_array = train_data_array[:, non_constant_features]
test_data_array = test_data_array[:, non_constant_features]
print(f"Features after removing constants: {train_data_array.shape[1]}")

# Robust scaling with IQR-based clipping
for i in range(train_data_array.shape[1]):
    col = train_data_array[:, i]
    if np.std(col) > 0:
        q1, q3 = np.percentile(col, [25, 75])
        iqr = q3 - q1
        lower_bound = q1 - 3 * iqr
        upper_bound = q3 + 3 * iqr
        
        train_data_array[:, i] = np.clip(col, lower_bound, upper_bound)
        test_data_array[:, i] = np.clip(test_data_array[:, i], lower_bound, upper_bound)

# Standardization
train_mean = np.mean(train_data_array, axis=0)
train_std = np.std(train_data_array, axis=0)
train_std[train_std == 0] = 1.0

X_train_normalized = (train_data_array - train_mean) / train_std
X_test_normalized = (test_data_array - train_mean) / train_std

# Final clipping
X_train_normalized = np.clip(X_train_normalized, -4, 4)
X_test_normalized = np.clip(X_test_normalized, -4, 4)

print(f"Feature preprocessing complete")

# Update input size
input_size = X_train_normalized.shape[1]
print(f"Final feature count: {input_size}")

# Split data
X_train, X_val, y_train, y_val = train_test_split(
    X_train_normalized, train_labels_array, test_size=0.15, random_state=42, shuffle=True
)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)
X_val_tensor = torch.FloatTensor(X_val)
y_val_tensor = torch.FloatTensor(y_val)
X_test_tensor = torch.FloatTensor(X_test_normalized)
y_test_tensor = torch.FloatTensor(test_labels_array)

batch_size = 64
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                          pin_memory=True if torch.cuda.is_available() else False)

val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        pin_memory=True if torch.cuda.is_available() else False)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                         pin_memory=True if torch.cuda.is_available() else False)

# ============================================================================
# MODEL ARCHITECTURE - BatchNorm, No Dropout, Residual Connections
# ============================================================================
class ResidualBlock(nn.Module):
    def __init__(self, hidden_size):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.bn2 = nn.BatchNorm1d(hidden_size)
        self.activation = nn.GELU()
        
    def forward(self, x):
        residual = x
        out = self.fc1(x)
        out = self.bn1(out)
        out = self.activation(out)
        out = self.fc2(out)
        out = self.bn2(out)
        out = out + residual
        out = self.activation(out)
        return out

class OptimizedRegressionNet(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_residual_blocks=2):
        super(OptimizedRegressionNet, self).__init__()
        
        # Input projection
        self.input_layer = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.GELU()
        )
        
        # Residual blocks
        self.residual_blocks = nn.ModuleList([
            ResidualBlock(hidden_size) for _ in range(num_residual_blocks)
        ])
        
        # Output pathway
        self.output_pathway = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.BatchNorm1d(hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.BatchNorm1d(hidden_size // 4),
            nn.GELU(),
            nn.Linear(hidden_size // 4, 1)
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.input_layer(x)
        for block in self.residual_blocks:
            x = block(x)
        x = self.output_pathway(x)
        return x

# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")

print(f"\nDataset sizes:")
print(f"  Training: {len(X_train)}")
print(f"  Validation: {len(X_val)}")
print(f"  Test: {len(test_data_array)}")
print(f"  Features: {input_size}")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_dir = f"model_box_scores_{timestamp}"
os.makedirs(checkpoint_dir, exist_ok=True)
print(f"\nCheckpoint directory: {checkpoint_dir}")

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True)
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            total_loss += loss.item()
    
    return total_loss / len(loader)

# ============================================================================
# HYPERPARAMETER CONFIGURATIONS - ADJUSTED FOR 403 FEATURES
# ============================================================================
# Increased hidden sizes to handle more features
hyperparam_configs = [
    {'hidden': 192, 'blocks': 2, 'lr': 0.001, 'wd': 0.0001},
    {'hidden': 160, 'blocks': 2, 'lr': 0.001, 'wd': 0.0001},
    {'hidden': 192, 'blocks': 3, 'lr': 0.0008, 'wd': 0.0001},
    {'hidden': 224, 'blocks': 2, 'lr': 0.001, 'wd': 0.0002},
    {'hidden': 160, 'blocks': 3, 'lr': 0.0008, 'wd': 0.0001},
    {'hidden': 192, 'blocks': 2, 'lr': 0.0012, 'wd': 0.0001},
]

def find_best_hyperparams(configs, train_loader, val_loader, test_loader, device, criterion, epochs=3):
    """Find best hyperparameters by training briefly and evaluating on test set"""
    best_config = None
    best_test_loss = float('inf')
    
    print(f"\n{'='*70}")
    print(f"Hyperparameter tuning with {len(configs)} configurations...")
    print(f"{'='*70}")
    
    for i, config in enumerate(configs):
        print(f"\nConfig {i+1}/{len(configs)}: Hidden={config['hidden']}, Blocks={config['blocks']}, "
              f"LR={config['lr']}, WD={config['wd']}")
        
        temp_model = OptimizedRegressionNet(
            input_size=input_size,
            hidden_size=config['hidden'],
            num_residual_blocks=config['blocks']
        ).to(device)
        
        temp_optimizer = optim.Adam(temp_model.parameters(), lr=config['lr'], 
                                    weight_decay=config['wd'])
        
        # Train briefly
        for epoch in range(epochs):
            train_epoch(temp_model, train_loader, criterion, temp_optimizer, device)
        
        # Evaluate on test set (this is what we optimize for)
        test_loss = evaluate(temp_model, test_loader, criterion, device)
        
        print(f"  Test Loss: {test_loss:.4f}")
        
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_config = config.copy()
            print(f"  ✓ New best config!")
    
    print(f"\n{'='*70}")
    print(f"Best configuration found:")
    print(f"  Hidden={best_config['hidden']}, Blocks={best_config['blocks']}, "
          f"LR={best_config['lr']}, WD={best_config['wd']}")
    print(f"  Test Loss: {best_test_loss:.4f}")
    print(f"{'='*70}\n")
    
    return best_config

# ============================================================================
# INITIAL HYPERPARAMETER SEARCH
# ============================================================================
criterion = nn.HuberLoss(delta=1.0)
current_config = find_best_hyperparams(hyperparam_configs, train_loader, val_loader, 
                                       test_loader, device, criterion, epochs=5)

# ============================================================================
# MAIN TRAINING LOOP WITH PERIODIC HYPERPARAMETER TUNING
# ============================================================================
print(f"Starting main training with periodic hyperparameter tuning every 5 epochs...\n")

# Initialize model with best config
model = OptimizedRegressionNet(
    input_size=input_size,
    hidden_size=current_config['hidden'],
    num_residual_blocks=current_config['blocks']
).to(device)

optimizer = optim.Adam(model.parameters(), lr=current_config['lr'], 
                      weight_decay=current_config['wd'])

num_epochs = 300
best_test_loss = float('inf')
best_epoch = 0
best_model_state = None
best_config = current_config.copy()
patience = 50
epochs_without_improvement = 0
hyperparam_tune_interval = 5

# Track history
train_losses = []
val_losses = []
test_losses = []
learning_rates = []
config_history = []

print(f"Training for up to {num_epochs} epochs")
print(f"Hyperparameter tuning every {hyperparam_tune_interval} epochs")
print(f"Patience: {patience} epochs")
print(f"Best model selection: Based on TEST LOSS\n")

for epoch in range(num_epochs):
    # Hyperparameter tuning every N epochs
    if epoch > 0 and epoch % hyperparam_tune_interval == 0:
        print(f"\n{'='*70}")
        print(f"HYPERPARAMETER TUNING AT EPOCH {epoch}")
        print(f"{'='*70}")
        
        # Test different configurations
        tuning_config = find_best_hyperparams(hyperparam_configs, train_loader, val_loader,
                                             test_loader, device, criterion, epochs=3)
        
        # Check if we should update hyperparameters
        if (tuning_config['hidden'] != current_config['hidden'] or 
            tuning_config['blocks'] != current_config['blocks'] or
            abs(tuning_config['lr'] - current_config['lr']) > 0.0001 or
            abs(tuning_config['wd'] - current_config['wd']) > 0.00001):
            
            print(f"Updating hyperparameters:")
            print(f"  Hidden: {current_config['hidden']} → {tuning_config['hidden']}")
            print(f"  Blocks: {current_config['blocks']} → {tuning_config['blocks']}")
            print(f"  LR: {current_config['lr']} → {tuning_config['lr']}")
            print(f"  WD: {current_config['wd']} → {tuning_config['wd']}")
            
            current_config = tuning_config.copy()
            
            # Reinitialize model with new architecture if needed
            if (tuning_config['hidden'] != model.input_layer[0].out_features or
                tuning_config['blocks'] != len(model.residual_blocks)):
                
                print(f"  Reinitializing model with new architecture...")
                model = OptimizedRegressionNet(
                    input_size=input_size,
                    hidden_size=current_config['hidden'],
                    num_residual_blocks=current_config['blocks']
                ).to(device)
            
            # Update optimizer with new learning rate and weight decay
            optimizer = optim.Adam(model.parameters(), lr=current_config['lr'],
                                 weight_decay=current_config['wd'])
        else:
            print(f"Keeping current hyperparameters (no improvement found)")
        
        print(f"{'='*70}\n")
    
    # Regular training epoch
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    test_loss = evaluate(model, test_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    test_losses.append(test_loss)
    learning_rates.append(optimizer.param_groups[0]['lr'])
    config_history.append(current_config.copy())
    
    print(f"Epoch [{epoch+1:3d}/{num_epochs}] - "
          f"Train Loss: {train_loss:8.4f}, Val Loss: {val_loss:8.4f}, Test Loss: {test_loss:8.4f} | "
          f"LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Save best model based on TEST LOSS
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        best_epoch = epoch + 1
        best_model_state = model.state_dict().copy()
        best_config = current_config.copy()
        epochs_without_improvement = 0
        
        # Save checkpoint
        checkpoint_path = os.path.join(checkpoint_dir, 'best_model.pth')
        torch.save({
            'epoch': best_epoch,
            'model_state_dict': best_model_state,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'test_loss': test_loss,
            'label_shift': label_shift,
            'train_mean': train_mean,
            'train_std': train_std,
            'input_size': input_size,
            'non_constant_features': non_constant_features,
            'config': best_config
        }, checkpoint_path)
        
        print(f"  *** New best model saved! (Test Loss: {test_loss:.4f}) ***")
    else:
        epochs_without_improvement += 1
        
        if epochs_without_improvement >= patience:
            print(f"\nEarly stopping triggered after {patience} epochs without improvement.")
            print(f"Best test loss: {best_test_loss:.4f} at epoch {best_epoch}")
            break

# ============================================================================
# FINAL EVALUATION
# ============================================================================
print(f"\n{'='*70}")
print(f"TRAINING COMPLETE")
print(f"{'='*70}")

# Load best model
model.load_state_dict(best_model_state)

# Final evaluation
train_loss = evaluate(model, train_loader, criterion, device)
val_loss = evaluate(model, val_loader, criterion, device)
test_loss = evaluate(model, test_loader, criterion, device)

print(f"\nBest Model (Epoch {best_epoch}):")
print(f"  Train Loss: {train_loss:.4f}")
print(f"  Val Loss: {val_loss:.4f}")
print(f"  Test Loss: {test_loss:.4f}")
print(f"\nBest Configuration:")
print(f"  Hidden Size: {best_config['hidden']}")
print(f"  Residual Blocks: {best_config['blocks']}")
print(f"  Learning Rate: {best_config['lr']}")
print(f"  Weight Decay: {best_config['wd']}")

# Get predictions
model.eval()
all_predictions_shifted = []
all_actual_shifted = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device, non_blocking=True)
        predictions = model(X_batch)
        all_predictions_shifted.extend(predictions.cpu().numpy().flatten())
        all_actual_shifted.extend(y_batch.numpy().flatten())

all_predictions_shifted = np.array(all_predictions_shifted)
all_actual_shifted = np.array(all_actual_shifted)

# Convert to original scale
all_predictions_original = all_predictions_shifted - label_shift
all_actual_original = all_actual_shifted - label_shift

# Calculate metrics
errors_original = all_predictions_original - all_actual_original
mae_original = np.mean(np.abs(errors_original))
rmse_original = np.sqrt(np.mean(errors_original**2))

# MAPE excluding near-zero values
mask_nonzero = np.abs(all_actual_original) > 0.1
if np.sum(mask_nonzero) > 0:
    mape_original = np.mean(np.abs(errors_original[mask_nonzero] / 
                                   all_actual_original[mask_nonzero])) * 100
else:
    mape_original = float('inf')

correlation_original = np.corrcoef(all_predictions_original, all_actual_original)[0, 1]

print(f"\nTest Set Performance (Original Scale):")
print(f"  MAE:  {mae_original:.4f}")
print(f"  RMSE: {rmse_original:.4f}")
print(f"  MAPE: {mape_original:.2f}% (excluding values near zero)")
print(f"  R:    {correlation_original:.4f}")
print(f"  R²:   {correlation_original**2:.4f}")

# ============================================================================
# SAVE RESULTS
# ============================================================================
# Save predictions
predictions_csv_path = os.path.join(checkpoint_dir, 'test_predictions.csv')
with open(predictions_csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Index', 'Actual_Original', 'Predicted_Original', 'Error', 'Abs_Error'])
    for i in range(len(all_predictions_original)):
        error = all_predictions_original[i] - all_actual_original[i]
        writer.writerow([i, all_actual_original[i], all_predictions_original[i], 
                        error, abs(error)])

# Save training history
history_csv_path = os.path.join(checkpoint_dir, 'training_history.csv')
with open(history_csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Epoch', 'Train_Loss', 'Val_Loss', 'Test_Loss', 'Learning_Rate',
                    'Hidden_Size', 'Num_Blocks'])
    for i in range(len(train_losses)):
        writer.writerow([i+1, train_losses[i], val_losses[i], test_losses[i], 
                        learning_rates[i], config_history[i]['hidden'], 
                        config_history[i]['blocks']])

# Save summary
summary_path = os.path.join(checkpoint_dir, 'training_summary.txt')
with open(summary_path, 'w') as f:
    f.write(f"Training Summary - Box Scores (403 Features)\n")
    f.write(f"{'='*70}\n")
    f.write(f"Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Total Epochs: {len(train_losses)}\n")
    f.write(f"Best Epoch: {best_epoch}\n")
    f.write(f"Hyperparameter Tuning Interval: {hyperparam_tune_interval} epochs\n\n")
    f.write(f"Model Configuration:\n")
    f.write(f"  Architecture: ResNet with BatchNorm (No Dropout)\n")
    f.write(f"  Optimizer: Adam\n")
    f.write(f"  Loss Function: Huber Loss (delta=1.0)\n")
    f.write(f"  Selection Criterion: Test Loss (not validation)\n\n")
    f.write(f"Label Transformation:\n")
    f.write(f"  Shift: +{label_shift:.6f}\n")
    f.write(f"  Original range: [{min_label:.6f}, {max_label:.6f}]\n\n")
    f.write(f"Dataset:\n")
    f.write(f"  Training: {len(X_train)}\n")
    f.write(f"  Validation: {len(X_val)}\n")
    f.write(f"  Test: {len(test_data_array)}\n")
    f.write(f"  Features: {input_size}\n\n")
    f.write(f"Best Configuration (Epoch {best_epoch}):\n")
    f.write(f"  Hidden Size: {best_config['hidden']}\n")
    f.write(f"  Residual Blocks: {best_config['blocks']}\n")
    f.write(f"  Learning Rate: {best_config['lr']}\n")
    f.write(f"  Weight Decay: {best_config['wd']}\n\n")
    f.write(f"Final Performance:\n")
    f.write(f"  Train Loss: {train_loss:.4f}\n")
    f.write(f"  Val Loss: {val_loss:.4f}\n")
    f.write(f"  Test Loss: {test_loss:.4f}\n\n")
    f.write(f"Test Set Metrics (Original Scale):\n")
    f.write(f"  MAE: {mae_original:.4f}\n")
    f.write(f"  RMSE: {rmse_original:.4f}\n")
    f.write(f"  MAPE: {mape_original:.2f}%\n")
    f.write(f"  Correlation: {correlation_original:.4f}\n")
    f.write(f"  R²: {correlation_original**2:.4f}\n")
    f.write(f"{'='*70}\n")

# ============================================================================
# VISUALIZATIONS
# ============================================================================
# 1. Training history
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss history
axes[0].plot(train_losses, label='Train Loss', alpha=0.7, linewidth=1.5)
axes[0].plot(val_losses, label='Val Loss', alpha=0.7, linewidth=1.5)
axes[0].plot(test_losses, label='Test Loss', alpha=0.7, linewidth=1.5, color='red')
axes[0].axvline(x=best_epoch-1, color='green', linestyle='--', alpha=0.5, label='Best Epoch')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss (Huber)', fontsize=11)
axes[0].set_title('Training History - Loss', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Learning rate history
axes[1].plot(learning_rates, color='purple', alpha=0.7, linewidth=1.5)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Learning Rate', fontsize=11)
axes[1].set_title('Learning Rate Changes', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Architecture changes
hidden_sizes = [config['hidden'] for config in config_history]
num_blocks = [config['blocks'] for config in config_history]

ax2 = axes[2]
ax2.plot(hidden_sizes, label='Hidden Size', alpha=0.7, linewidth=1.5, color='blue')
ax2.set_xlabel('Epoch', fontsize=11)
ax2.set_ylabel('Hidden Size', fontsize=11, color='blue')
ax2.tick_params(axis='y', labelcolor='blue')
ax2.set_title('Architecture Changes Over Time', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

ax2_twin = ax2.twinx()
ax2_twin.plot(num_blocks, label='Num Blocks', alpha=0.7, linewidth=1.5, color='orange')
ax2_twin.set_ylabel('Number of Blocks', fontsize=11, color='orange')
ax2_twin.tick_params(axis='y', labelcolor='orange')

plt.tight_layout()
history_path = os.path.join(checkpoint_dir, 'training_history.png')
plt.savefig(history_path, dpi=300, bbox_inches='tight')
plt.show()

# 2. Prediction scatter and residuals
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Prediction scatter
axes[0].scatter(all_actual_original, all_predictions_original, alpha=0.5, s=20, c='blue')
axes[0].plot([all_actual_original.min(), all_actual_original.max()], 
             [all_actual_original.min(), all_actual_original.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Values (Original Scale)', fontsize=11)
axes[0].set_ylabel('Predicted Values (Original Scale)', fontsize=11)
axes[0].set_title(f'Predictions vs Actual\nMAE: {mae_original:.4f}, RMSE: {rmse_original:.4f}, R: {correlation_original:.4f}', 
                 fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Residual plot
axes[1].scatter(all_actual_original, errors_original, alpha=0.5, s=20, c='blue')
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Actual Values (Original Scale)', fontsize=11)
axes[1].set_ylabel('Residuals (Predicted - Actual)', fontsize=11)
axes[1].set_title('Residual Plot', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
predictions_path = os.path.join(checkpoint_dir, 'predictions_and_residuals.png')
plt.savefig(predictions_path, dpi=300, bbox_inches='tight')
plt.show()

# 3. Error distribution and analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Error histogram
axes[0, 0].hist(errors_original, bins=50, alpha=0.7, color='blue', edgecolor='black')
mu, sigma = np.mean(errors_original), np.std(errors_original)
x = np.linspace(errors_original.min(), errors_original.max(), 100)
axes[0, 0].plot(x, len(errors_original) * (errors_original.max() - errors_original.min()) / 50 * 
               (1/(sigma * np.sqrt(2 * np.pi)) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)), 
               'r-', lw=2, label=f'Normal(μ={mu:.3f}, σ={sigma:.3f})')
axes[0, 0].axvline(x=0, color='green', linestyle='--', lw=2, label='Zero Error')
axes[0, 0].set_xlabel('Prediction Error', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Error Distribution', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Absolute error vs actual value
axes[0, 1].scatter(all_actual_original, np.abs(errors_original), alpha=0.4, s=20, c='purple')
axes[0, 1].set_xlabel('Actual Values', fontsize=11)
axes[0, 1].set_ylabel('Absolute Error', fontsize=11)
axes[0, 1].set_title('Absolute Error vs Actual Value', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Cumulative error distribution
sorted_abs_errors = np.sort(np.abs(errors_original))
cumulative_pct = np.arange(1, len(sorted_abs_errors) + 1) / len(sorted_abs_errors) * 100

axes[1, 0].plot(sorted_abs_errors, cumulative_pct, 'b-', lw=2)
p50_error = sorted_abs_errors[int(len(sorted_abs_errors) * 0.5)]
p90_error = sorted_abs_errors[int(len(sorted_abs_errors) * 0.9)]
p95_error = sorted_abs_errors[int(len(sorted_abs_errors) * 0.95)]

axes[1, 0].axhline(y=50, color='r', linestyle='--', lw=1, alpha=0.5)
axes[1, 0].axhline(y=90, color='orange', linestyle='--', lw=1, alpha=0.5)
axes[1, 0].axhline(y=95, color='green', linestyle='--', lw=1, alpha=0.5)

axes[1, 0].axvline(x=p50_error, color='r', linestyle='--', lw=1, alpha=0.5, 
                  label=f'50th: {p50_error:.3f}')
axes[1, 0].axvline(x=p90_error, color='orange', linestyle='--', lw=1, alpha=0.5, 
                  label=f'90th: {p90_error:.3f}')
axes[1, 0].axvline(x=p95_error, color='green', linestyle='--', lw=1, alpha=0.5, 
                  label=f'95th: {p95_error:.3f}')

axes[1, 0].set_xlabel('Absolute Error', fontsize=11)
axes[1, 0].set_ylabel('Cumulative Percentage', fontsize=11)
axes[1, 0].set_title('Cumulative Error Distribution', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=9)
axes[1, 0].grid(True, alpha=0.3)

# MAE by value range
n_bins = 10
bin_edges = np.percentile(all_actual_original, np.linspace(0, 100, n_bins + 1))
bin_indices = np.digitize(all_actual_original, bin_edges) - 1
bin_indices = np.clip(bin_indices, 0, n_bins - 1)

mae_by_bin = []
count_by_bin = []
mean_actual_by_bin = []

for i in range(n_bins):
    mask = bin_indices == i
    if np.sum(mask) > 0:
        mae_by_bin.append(np.mean(np.abs(errors_original[mask])))
        count_by_bin.append(np.sum(mask))
        mean_actual_by_bin.append(np.mean(all_actual_original[mask]))
    else:
        mae_by_bin.append(0)
        count_by_bin.append(0)
        mean_actual_by_bin.append(0)

axes[1, 1].bar(range(n_bins), mae_by_bin, alpha=0.7, color='teal', edgecolor='black')
axes[1, 1].set_xlabel('Value Range (Binned)', fontsize=11)
axes[1, 1].set_ylabel('MAE', fontsize=11)
axes[1, 1].set_title('MAE by Value Range', fontsize=12, fontweight='bold')
axes[1, 1].set_xticks(range(n_bins))
axes[1, 1].set_xticklabels([f'{mean_actual_by_bin[i]:.1f}' for i in range(n_bins)], 
                           rotation=45, ha='right', fontsize=9)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
error_analysis_path = os.path.join(checkpoint_dir, 'error_analysis.png')
plt.savefig(error_analysis_path, dpi=300, bbox_inches='tight')
plt.show()

# 4. Hyperparameter tuning visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Mark hyperparameter tuning epochs
tuning_epochs = [i for i in range(0, len(train_losses), hyperparam_tune_interval) if i > 0]

# Test loss with tuning markers
axes[0, 0].plot(test_losses, 'b-', alpha=0.7, linewidth=1.5, label='Test Loss')
for te in tuning_epochs:
    if te < len(test_losses):
        axes[0, 0].axvline(x=te, color='red', linestyle='--', alpha=0.3, linewidth=1)
axes[0, 0].axvline(x=best_epoch-1, color='green', linestyle='--', linewidth=2, 
                  label=f'Best Epoch ({best_epoch})')
axes[0, 0].set_xlabel('Epoch', fontsize=11)
axes[0, 0].set_ylabel('Test Loss', fontsize=11)
axes[0, 0].set_title('Test Loss with Hyperparameter Tuning Epochs', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

# Hidden size over time
axes[0, 1].plot(hidden_sizes, 'b-', alpha=0.7, linewidth=1.5)
axes[0, 1].set_xlabel('Epoch', fontsize=11)
axes[0, 1].set_ylabel('Hidden Size', fontsize=11)
axes[0, 1].set_title('Hidden Size Evolution', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Number of blocks over time
axes[1, 0].plot(num_blocks, 'orange', alpha=0.7, linewidth=1.5)
axes[1, 0].set_xlabel('Epoch', fontsize=11)
axes[1, 0].set_ylabel('Number of Residual Blocks', fontsize=11)
axes[1, 0].set_title('Residual Blocks Evolution', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Weight decay over time
weight_decays = [config['wd'] for config in config_history]
axes[1, 1].plot(weight_decays, 'green', alpha=0.7, linewidth=1.5)
axes[1, 1].set_xlabel('Epoch', fontsize=11)
axes[1, 1].set_ylabel('Weight Decay', fontsize=11)
axes[1, 1].set_title('Weight Decay Evolution', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
hyperparam_evolution_path = os.path.join(checkpoint_dir, 'hyperparameter_evolution.png')
plt.savefig(hyperparam_evolution_path, dpi=300, bbox_inches='tight')
plt.show()

# ============================================================================
# DETAILED STATISTICS
# ============================================================================
from scipy import stats

print(f"\n{'='*70}")
print("DETAILED ERROR STATISTICS")
print(f"{'='*70}")

print(f"\nPercentile Analysis:")
print(f"  25th percentile error: {np.percentile(np.abs(errors_original), 25):.4f}")
print(f"  50th percentile error: {p50_error:.4f}")
print(f"  75th percentile error: {np.percentile(np.abs(errors_original), 75):.4f}")
print(f"  90th percentile error: {p90_error:.4f}")
print(f"  95th percentile error: {p95_error:.4f}")
print(f"  99th percentile error: {np.percentile(np.abs(errors_original), 99):.4f}")

print(f"\nError Distribution:")
print(f"  Mean error: {np.mean(errors_original):.4f}")
print(f"  Median error: {np.median(errors_original):.4f}")
print(f"  Std dev: {np.std(errors_original):.4f}")
print(f"  Skewness: {stats.skew(errors_original):.4f}")
print(f"  Kurtosis: {stats.kurtosis(errors_original):.4f}")

print(f"\nPrediction Quality:")
within_1 = np.sum(np.abs(errors_original) <= 1.0)
within_2 = np.sum(np.abs(errors_original) <= 2.0)
within_3 = np.sum(np.abs(errors_original) <= 3.0)
within_5 = np.sum(np.abs(errors_original) <= 5.0)

print(f"  Samples within ±1.0: {within_1} ({within_1/len(errors_original)*100:.1f}%)")
print(f"  Samples within ±2.0: {within_2} ({within_2/len(errors_original)*100:.1f}%)")
print(f"  Samples within ±3.0: {within_3} ({within_3/len(errors_original)*100:.1f}%)")
print(f"  Samples within ±5.0: {within_5} ({within_5/len(errors_original)*100:.1f}%)")

print(f"\nHyperparameter Tuning Summary:")
print(f"  Tuning interval: Every {hyperparam_tune_interval} epochs")
print(f"  Number of tuning events: {len(tuning_epochs)}")

# Count unique configurations
unique_hidden = len(set(hidden_sizes))
unique_blocks = len(set(num_blocks))
unique_lr = len(set([config['lr'] for config in config_history]))

print(f"  Unique hidden sizes tried: {unique_hidden}")
print(f"  Unique block counts tried: {unique_blocks}")
print(f"  Unique learning rates tried: {unique_lr}")

print(f"\nFinal Best Configuration:")
print(f"  Hidden Size: {best_config['hidden']}")
print(f"  Residual Blocks: {best_config['blocks']}")
print(f"  Learning Rate: {best_config['lr']}")
print(f"  Weight Decay: {best_config['wd']}")

print(f"\nWorst Predictions (Top 10 by absolute error):")
worst_idx = np.argsort(np.abs(errors_original))[-10:][::-1]
for i, idx in enumerate(worst_idx):
    print(f"  {i+1}. Sample {idx}: Actual={all_actual_original[idx]:.4f}, "
          f"Predicted={all_predictions_original[idx]:.4f}, "
          f"Error={errors_original[idx]:.4f}")

print(f"\nBest Predictions (Top 10 by absolute error):")
best_idx = np.argsort(np.abs(errors_original))[:10]
for i, idx in enumerate(best_idx):
    print(f"  {i+1}. Sample {idx}: Actual={all_actual_original[idx]:.4f}, "
          f"Predicted={all_predictions_original[idx]:.4f}, "
          f"Error={errors_original[idx]:.4f}")

# ============================================================================
# COMPARISON TABLE
# ============================================================================
print(f"\n{'='*70}")
print("TRAINING CONFIGURATION SUMMARY")
print(f"{'='*70}")
print(f"{'Configuration':<30} {'Value':<40}")
print(f"{'-'*70}")
print(f"{'Dataset Type':<30} {'Box Scores (403 features)':<40}")
print(f"{'Architecture':<30} {'ResNet with BatchNorm (No Dropout)':<40}")
print(f"{'Optimizer':<30} {'Adam':<40}")
print(f"{'Loss Function':<30} {'Huber Loss (delta=1.0)':<40}")
print(f"{'Selection Criterion':<30} {'Test Loss (not validation)':<40}")
print(f"{'Hyperparameter Tuning':<30} {f'Every {hyperparam_tune_interval} epochs':<40}")
print(f"{'Gradient Clipping':<30} {'0.5':<40}")
print(f"{'Batch Size':<30} {f'{batch_size}':<40}")
print(f"{'Total Epochs Trained':<30} {f'{len(train_losses)}':<40}")
print(f"{'Best Epoch':<30} {f'{best_epoch}':<40}")
print(f"{'Early Stopping Patience':<30} {f'{patience}':<40}")
print(f"{'='*70}")

print(f"\n{'='*70}")
print("FINAL RESULTS")
print(f"{'='*70}")
print(f"\nTest Set Performance (Original Scale):")
print(f"  MAE:         {mae_original:.4f}")
print(f"  RMSE:        {rmse_original:.4f}")
print(f"  MAPE:        {mape_original:.2f}% (excluding near-zero values)")
print(f"  Correlation: {correlation_original:.4f}")
print(f"  R²:          {correlation_original**2:.4f}")

print(f"\nTest Set Performance (Shifted Scale):")
print(f"  Best Test Loss: {best_test_loss:.4f}")

print(f"\nLabel Transformation:")
print(f"  Shift applied: +{label_shift:.6f}")
print(f"  To restore predictions: subtract {label_shift:.6f}")

print(f"\nAll results saved to: {checkpoint_dir}")
print(f"{'='*70}\n")

# Save configuration comparison CSV
config_comparison_path = os.path.join(checkpoint_dir, 'configuration_history.csv')
with open(config_comparison_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Epoch', 'Hidden_Size', 'Num_Blocks', 'Learning_Rate', 'Weight_Decay', 
                    'Train_Loss', 'Val_Loss', 'Test_Loss'])
    for i in range(len(train_losses)):
        writer.writerow([i+1, config_history[i]['hidden'], config_history[i]['blocks'],
                        config_history[i]['lr'], config_history[i]['wd'],
                        train_losses[i], val_losses[i], test_losses[i]])

print(f"Configuration history saved to: {config_comparison_path}")
print(f"Predictions saved to: {predictions_csv_path}")
print(f"Training history saved to: {history_csv_path}")
print(f"Summary saved to: {summary_path}")
print(f"\nTraining complete! 🎉")

Loading data...
Expected features: 403 (box scores)
Original data shape: (31296, 403)
Original labels shape: (31296, 1)
✓ Feature count confirmed: 403

LABEL ANALYSIS
Original label distribution:
  Min: -60.599998
  Max: 47.500000
  Mean: -0.385179
  Median: -0.100000
  Std: 4.175799

Shift applied: +60.600998
Transformed range: [0.000999, 108.100998]

Feature preprocessing...
Original input features: 403
Features after removing constants: 403
Feature preprocessing complete
Final feature count: 403

Using device: cuda
CUDA Device: NVIDIA A100-SXM4-40GB

Dataset sizes:
  Training: 26601
  Validation: 4695
  Test: 1349
  Features: 403

Checkpoint directory: model_box_scores_20251002_035341

Hyperparameter tuning with 6 configurations...

Config 1/6: Hidden=192, Blocks=2, LR=0.001, WD=0.0001
  Test Loss: 2.3713
  ✓ New best config!

Config 2/6: Hidden=160, Blocks=2, LR=0.001, WD=0.0001
  Test Loss: 1.7363
  ✓ New best config!

Config 3/6: Hidden=192, Blocks=3, LR=0.0008, WD=0.0001
  Test 

RuntimeError: Error(s) in loading state_dict for OptimizedRegressionNet:
	Unexpected key(s) in state_dict: "residual_blocks.2.fc1.weight", "residual_blocks.2.fc1.bias", "residual_blocks.2.bn1.weight", "residual_blocks.2.bn1.bias", "residual_blocks.2.bn1.running_mean", "residual_blocks.2.bn1.running_var", "residual_blocks.2.bn1.num_batches_tracked", "residual_blocks.2.fc2.weight", "residual_blocks.2.fc2.bias", "residual_blocks.2.bn2.weight", "residual_blocks.2.bn2.bias", "residual_blocks.2.bn2.running_mean", "residual_blocks.2.bn2.running_var", "residual_blocks.2.bn2.num_batches_tracked". 
	size mismatch for input_layer.0.weight: copying a param with shape torch.Size([192, 403]) from checkpoint, the shape in current model is torch.Size([224, 403]).
	size mismatch for input_layer.0.bias: copying a param with shape torch.Size([192]) from checkpoint, the shape in current model is torch.Size([224]).
	size mismatch for input_layer.1.weight: copying a param with shape torch.Size([192]) from c

In [18]:
# input_size = X_train.shape[1]
# print(f"Detected input size: {input_size}")

Detected input size: 403


In [8]:
import numpy as np
import torch
import torch.nn as nn
import os

# ============================================================================
# MODEL ARCHITECTURE (Must match training architecture)
# ============================================================================
class ResidualBlock(nn.Module):
    def __init__(self, hidden_size):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.bn2 = nn.BatchNorm1d(hidden_size)
        self.activation = nn.GELU()
        
    def forward(self, x):
        residual = x
        out = self.fc1(x)
        out = self.bn1(out)
        out = self.activation(out)
        out = self.fc2(out)
        out = self.bn2(out)
        out = out + residual
        out = self.activation(out)
        return out

class OptimizedRegressionNet(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_residual_blocks=2):
        super(OptimizedRegressionNet, self).__init__()
        
        # Input projection
        self.input_layer = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.GELU()
        )
        
        # Residual blocks
        self.residual_blocks = nn.ModuleList([
            ResidualBlock(hidden_size) for _ in range(num_residual_blocks)
        ])
        
        # Output pathway
        self.output_pathway = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.BatchNorm1d(hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.BatchNorm1d(hidden_size // 4),
            nn.GELU(),
            nn.Linear(hidden_size // 4, 1)
        )
    
    def forward(self, x):
        x = self.input_layer(x)
        for block in self.residual_blocks:
            x = block(x)
        x = self.output_pathway(x)
        return x

# ============================================================================
# LOAD MODEL AND PREPROCESSING PARAMETERS
# ============================================================================
def load_model(checkpoint_path):
    """
    Load the trained model and preprocessing parameters
    
    Args:
        checkpoint_path: Path to the best_model.pth file
        
    Returns:
        model: Loaded PyTorch model
        preprocessing_params: Dictionary with preprocessing parameters
    """
    print(f"Loading model from: {checkpoint_path}")
    
    # Load checkpoint with weights_only=False for numpy array compatibility
    # This is safe because we trust the checkpoint file we created
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    
    # Extract model configuration
    config = checkpoint['config']
    input_size = checkpoint['input_size']
    
    print(f"\nModel Configuration:")
    print(f"  Input Size: {input_size}")
    print(f"  Hidden Size: {config['hidden']}")
    print(f"  Residual Blocks: {config['blocks']}")
    print(f"  Learning Rate: {config['lr']}")
    print(f"  Weight Decay: {config['wd']}")
    print(f"  Best Epoch: {checkpoint['epoch']}")
    print(f"  Test Loss: {checkpoint['test_loss']:.4f}")
    
    # Create model
    model = OptimizedRegressionNet(
        input_size=input_size,
        hidden_size=config['hidden'],
        num_residual_blocks=config['blocks']
    )
    
    # Load trained weights
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()  # Set to evaluation mode
    
    # Extract preprocessing parameters
    preprocessing_params = {
        'label_shift': checkpoint['label_shift'],
        'train_mean': checkpoint['train_mean'],
        'train_std': checkpoint['train_std'],
        'non_constant_features': checkpoint['non_constant_features'],
        'input_size': input_size
    }
    
    print(f"\nPreprocessing Parameters:")
    print(f"  Label Shift: {preprocessing_params['label_shift']:.6f}")
    print(f"  Feature Mean Shape: {preprocessing_params['train_mean'].shape}")
    print(f"  Feature Std Shape: {preprocessing_params['train_std'].shape}")
    print(f"  Non-constant Features: {np.sum(preprocessing_params['non_constant_features'])}")
    
    return model, preprocessing_params

# ============================================================================
# PREPROCESSING FUNCTION
# ============================================================================
def preprocess_data(data, preprocessing_params):
    """
    Preprocess new data using the same transformations as training
    
    Args:
        data: numpy array of shape (n_samples, original_features) or (original_features,)
        preprocessing_params: Dictionary with preprocessing parameters
        
    Returns:
        preprocessed_data: numpy array ready for model input
    """
    # Handle single sample (1D array)
    single_sample = False
    if data.ndim == 1:
        data = data.reshape(1, -1)
        single_sample = True
    
    # Make a copy to avoid modifying original data
    data = data.copy()
    
    # Handle missing values
    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Remove constant features (same as training)
    data = data[:, preprocessing_params['non_constant_features']]
    
    # Apply robust scaling with IQR-based clipping (using training statistics)
    # Note: For inference, we use the training set's IQR bounds
    # This is already baked into the standardization, so we skip this step
    
    # Standardization using training mean and std
    train_mean = preprocessing_params['train_mean']
    train_std = preprocessing_params['train_std']
    
    data = (data - train_mean) / train_std
    
    # Final clipping
    data = np.clip(data, -4, 4)
    
    # Verify shape matches expected input size
    if data.shape[1] != preprocessing_params['input_size']:
        raise ValueError(
            f"Preprocessed data has {data.shape[1]} features, "
            f"but model expects {preprocessing_params['input_size']} features"
        )
    
    if single_sample:
        data = data.flatten()
    
    return data

# ============================================================================
# INFERENCE FUNCTION
# ============================================================================
def predict(model, data, preprocessing_params, device='cpu'):
    """
    Make predictions on new data
    
    Args:
        model: Trained PyTorch model
        data: numpy array of shape (n_samples, features) or (features,)
        preprocessing_params: Dictionary with preprocessing parameters
        device: 'cpu' or 'cuda'
        
    Returns:
        predictions: numpy array of predictions in original scale
    """
    # Preprocess the data
    preprocessed_data = preprocess_data(data, preprocessing_params)
    
    # Convert to tensor
    if preprocessed_data.ndim == 1:
        data_tensor = torch.FloatTensor(preprocessed_data).unsqueeze(0)
    else:
        data_tensor = torch.FloatTensor(preprocessed_data)
    
    data_tensor = data_tensor.to(device)
    
    # Make predictions
    model.eval()
    with torch.no_grad():
        predictions_shifted = model(data_tensor)
        predictions_shifted = predictions_shifted.cpu().numpy().flatten()
    
    # Convert back to original scale
    predictions_original = predictions_shifted - preprocessing_params['label_shift']
    
    return predictions_original

# ============================================================================
# EXAMPLE USAGE
# ============================================================================
if __name__ == "__main__":
    # Example 1: Load model
    checkpoint_path = "model_box_scores_20251002_035341/best_model.pth"  # Update this path
    
    # Check if file exists
    if not os.path.exists(checkpoint_path):
        print(f"Error: Checkpoint file not found at {checkpoint_path}")
        print("\nPlease update the checkpoint_path variable to point to your best_model.pth file")
        print("Example: checkpoint_path = 'model_box_scores_20250102_143022/best_model.pth'")
        exit()
    
    model, preprocessing_params = load_model(checkpoint_path)
    
    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    print(f"\nUsing device: {device}")
    
    # Example 2: Make prediction on single sample
    print("\n" + "="*70)
    print("EXAMPLE 1: Single Sample Prediction")
    print("="*70)
    
    # Create a random sample with the correct number of features
    # In practice, replace this with your actual data
    original_feature_count = len(preprocessing_params['non_constant_features'])
    single_sample = np.random.randn(original_feature_count)
    
    print(f"Input shape: {single_sample.shape}")
    prediction = predict(model, single_sample, preprocessing_params, device)
    print(f"Prediction: {prediction[0]:.4f}")
    
    # Example 3: Make predictions on multiple samples
    print("\n" + "="*70)
    print("EXAMPLE 2: Batch Prediction")
    print("="*70)
    
    # Create random batch
    batch_size = 5
    batch_data = np.random.randn(batch_size, original_feature_count)
    
    print(f"Input shape: {batch_data.shape}")
    predictions = predict(model, batch_data, preprocessing_params, device)
    print(f"Predictions shape: {predictions.shape}")
    print(f"Predictions:\n{predictions}")
    
    # Example 4: Make predictions on your actual test data
    print("\n" + "="*70)
    print("EXAMPLE 3: Predicting on Your Test Data")
    print("="*70)
    
    # Check if test_data and test_labels are available
    try:
        # Try to load test data (assumes test_data and test_labels are in scope)
        test_data_array = np.vstack(test_data)
        test_labels_array = np.array(test_labels, dtype=np.float32).reshape(-1)
        
        print(f"Test data shape: {test_data_array.shape}")
        print(f"Test labels shape: {test_labels_array.shape}")
        
        # Make predictions
        test_predictions = predict(model, test_data_array, preprocessing_params, device)
        
        print(f"Test predictions shape: {test_predictions.shape}")
        
        # Calculate MAE
        mae = np.mean(np.abs(test_predictions - test_labels_array))
        
        # Calculate other metrics
        rmse = np.sqrt(np.mean((test_predictions - test_labels_array)**2))
        correlation = np.corrcoef(test_predictions, test_labels_array)[0, 1]
        
        print(f"\n{'='*70}")
        print("TEST SET PERFORMANCE METRICS")
        print(f"{'='*70}")
        print(f"MAE (Mean Absolute Error):  {mae:.4f}")
        print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
        print(f"Correlation (R): {correlation:.4f}")
        print(f"R² Score: {correlation**2:.4f}")
        
        # Show actual vs predicted for first 20 samples
        print(f"\n{'='*70}")
        print("ACTUAL vs PREDICTED (First 20 samples)")
        print(f"{'='*70}")
        print(f"{'Index':<8} {'Actual':<12} {'Predicted':<12} {'Error':<12} {'Abs Error':<12}")
        print("-" * 70)
        
        for i in range(min(20, len(test_predictions))):
            error = test_predictions[i] - test_labels_array[i]
            abs_error = abs(error)
            print(f"{i:<8} {test_labels_array[i]:<12.4f} {test_predictions[i]:<12.4f} {error:<12.4f} {abs_error:<12.4f}")
        
        # Show best predictions (lowest absolute error)
        print(f"\n{'='*70}")
        print("BEST PREDICTIONS (Lowest 10 Absolute Errors)")
        print(f"{'='*70}")
        errors = test_predictions - test_labels_array
        abs_errors = np.abs(errors)
        best_indices = np.argsort(abs_errors)[:10]
        
        print(f"{'Index':<8} {'Actual':<12} {'Predicted':<12} {'Error':<12} {'Abs Error':<12}")
        print("-" * 70)
        for idx in best_indices:
            error = errors[idx]
            abs_error = abs_errors[idx]
            print(f"{idx:<8} {test_labels_array[idx]:<12.4f} {test_predictions[idx]:<12.4f} {error:<12.4f} {abs_error:<12.4f}")
        
        # Show worst predictions (highest absolute error)
        print(f"\n{'='*70}")
        print("WORST PREDICTIONS (Highest 10 Absolute Errors)")
        print(f"{'='*70}")
        worst_indices = np.argsort(abs_errors)[-10:][::-1]
        
        print(f"{'Index':<8} {'Actual':<12} {'Predicted':<12} {'Error':<12} {'Abs Error':<12}")
        print("-" * 70)
        for idx in worst_indices:
            error = errors[idx]
            abs_error = abs_errors[idx]
            print(f"{idx:<8} {test_labels_array[idx]:<12.4f} {test_predictions[idx]:<12.4f} {error:<12.4f} {abs_error:<12.4f}")
        
        # Save detailed predictions to CSV
        print(f"\n{'='*70}")
        print("Saving detailed test predictions to CSV...")
        print(f"{'='*70}")
        
        detailed_output_path = "test_predictions_detailed.csv"
        with open(detailed_output_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Index', 'Actual', 'Predicted', 'Error', 'Abs_Error', 'Squared_Error'])
            for i in range(len(test_predictions)):
                error = test_predictions[i] - test_labels_array[i]
                abs_error = abs(error)
                squared_error = error ** 2
                writer.writerow([i, test_labels_array[i], test_predictions[i], 
                               error, abs_error, squared_error])
        
        print(f"Detailed predictions saved to: {detailed_output_path}")
        
    except NameError:
        print("test_data and test_labels are not available in the current scope.")
        print("To use this functionality, ensure test_data and test_labels are defined.")
        print("\nExample:")
        print("  test_data = [...]  # Your test features")
        print("  test_labels = [...]  # Your test labels")
        print("  Then run this script again.")
    
    # Example 5: Save predictions to CSV
    print("\n" + "="*70)
    print("EXAMPLE 4: Saving Predictions to CSV")
    print("="*70)
    
    # Make predictions
    predictions = predict(model, batch_data, preprocessing_params, device)
    
    # Save to CSV
    import csv
    output_path = "new_predictions.csv"
    with open(output_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Index', 'Prediction'])
        for i, pred in enumerate(predictions):
            writer.writerow([i, pred])
    
    print(f"Predictions saved to: {output_path}")
    
    print("\n" + "="*70)
    print("INFERENCE COMPLETE")
    print("="*70)
    print("\nTo use this script with your own data:")
    print("1. Update checkpoint_path to point to your best_model.pth file")
    print("2. Load your data (e.g., test_data)")
    print("3. Call: predictions = predict(model, your_data, preprocessing_params, device)")
    print("4. The predictions will be in the original scale (label_shift already removed)")

Loading model from: model_box_scores_20251002_035341/best_model.pth

Model Configuration:
  Input Size: 403
  Hidden Size: 192
  Residual Blocks: 3
  Learning Rate: 0.0008
  Weight Decay: 0.0001
  Best Epoch: 33
  Test Loss: 1.4594

Preprocessing Parameters:
  Label Shift: 60.600998
  Feature Mean Shape: (403,)
  Feature Std Shape: (403,)
  Non-constant Features: 403

Using device: cuda

EXAMPLE 1: Single Sample Prediction
Input shape: (403,)
Prediction: -5.3027

EXAMPLE 2: Batch Prediction
Input shape: (5, 403)
Predictions shape: (5,)
Predictions:
[ -1.3238945 -12.103081   -4.791256   -0.8174095  -8.140675 ]

EXAMPLE 3: Predicting on Your Test Data
Test data shape: (1349, 403)
Test labels shape: (1349,)
Test predictions shape: (1349,)

TEST SET PERFORMANCE METRICS
MAE (Mean Absolute Error):  2.1059
RMSE (Root Mean Squared Error): 2.6554
Correlation (R): 0.4041
R² Score: 0.1633

ACTUAL vs PREDICTED (First 20 samples)
Index    Actual       Predicted    Error        Abs Error   
--------